# P5 — Read PyTorch in Colab (CPU)

No TODO functions. Run top to bottom and change only `hidden_width` from 4 to 6, then restore 4. This is an optional companion to P5, not a numbered Lab.


In [ ]:
import io
import torch
from torch import nn

torch.manual_seed(13)
torch.set_num_threads(1)
hidden_width = 4  # edit only this value to 6, then reset to 4

class SmallClassifier(nn.Module):
    def __init__(self, hidden=4):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(3, hidden), nn.ReLU(), nn.Linear(hidden, 2))
    def forward(self, x):
        return self.layers(x)

model = SmallClassifier(hidden_width)
features = torch.tensor([[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]], dtype=torch.float32)
targets = torch.tensor([0, 1], dtype=torch.long)
logits = model(features)
print("input:", tuple(features.shape), features.dtype)
print("output:", tuple(logits.shape), logits.dtype)
print("targets:", tuple(targets.shape), targets.dtype)
assert logits.shape == (2, 2)


## Five lines to read, not memorize

1. `zero_grad` clears old gradients. 2. `model(features)` calls `forward`. 3. The loss compares outputs to class-index targets. 4. `backward` computes gradients. 5. `step` updates parameters. Run twice to see a loss change; changing hidden width changes parameter shapes.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
before = float(criterion(model(features), targets))
for _ in range(2):
    optimizer.zero_grad()
    predictions = model(features)
    loss = criterion(predictions, targets)
    loss.backward()
    optimizer.step()
after = float(criterion(model(features), targets))
print("loss before/after:", before, after)
assert torch.isfinite(torch.tensor(after))


## State is not the class definition

`state_dict()` maps parameter names to tensors. Save and reload in memory here; no credentials, external datasets, local file or GPU required. Restore `hidden_width=4` and rerun all cells before moving on.


In [ ]:
keys = sorted(model.state_dict())
print("state_dict keys:", keys)
buffer = io.BytesIO()
torch.save(model.state_dict(), buffer)
buffer.seek(0)
restored = SmallClassifier(hidden_width)
restored.load_state_dict(torch.load(buffer, map_location="cpu", weights_only=True))
restored.eval()
model.eval()
with torch.no_grad():
    assert torch.allclose(restored(features), model(features))
print("PASS: state_dict round trip")
